# Notebook 15 — Confidence-Based Fusion: CRNN + TrOCR + Lexicon

In [1]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import time
from dataclasses import dataclass
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

DEVICE = (torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))
print(f"Using device: {DEVICE}")

Using device: mps


In [2]:
@dataclass
class Config:
    data_root: Path = Path("../data/pharmacy_lk")
    val_csv: str = "splits/val.csv"; test_csv: str = "splits/test.csv"; train_csv: str = "splits/train.csv"
    img_dir: str = "images"; img_col: str = "image_filename"; label_col: str = "medicine_name"
    img_height: int = 48; img_width: int = 320
    rnn_hidden: int = 256; rnn_layers: int = 2; dropout: float = 0.2
    crnn_ckpt: Path = Path("../checkpoints/baseline_crnn_v2/best.pt")
    trocr_ckpt: Path = Path("../checkpoints/benchmark_trocr/best")
    processor_name: str = "microsoft/trocr-small-handwritten"
    batch_size: int = 8; max_target_len: int = 24; num_workers: int = 0
    crnn_lex_tau: float = 0.6
    out_dir: Path = Path("../checkpoints/fusion")

CFG = Config()
CFG.out_dir.mkdir(parents=True, exist_ok=True)

## 1. Metrics + lexicon

In [3]:
def edit_distance(a, b):
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        curr = [i]
        for j, cb in enumerate(b, 1):
            curr.append(min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = curr
    return prev[-1]

def corpus_metrics(preds, refs):
    tot = sum(edit_distance(p, r) for p, r in zip(preds, refs))
    chars = sum(len(r) for r in refs); exact = sum(p == r for p, r in zip(preds, refs))
    return {"CER": tot / max(chars, 1), "ExactMatch": exact / len(refs), "n": len(refs)}

lexicon = set(pd.read_csv(CFG.data_root / CFG.train_csv)[CFG.label_col].astype(str).str.strip().str.lower())
by_len = defaultdict(list)
for w in sorted(lexicon): by_len[len(w)].append(w)
LEXSET = lexicon
print(f"lexicon size: {len(lexicon)}")

def nearest(word, gap=3):
    if not word: return None, 10**9
    if word in by_len.get(len(word), ()): return word, 0
    best, bd = None, 10**9
    for L in range(len(word)-gap, len(word)+gap+1):
        for e in by_len.get(L, ()):
            d = edit_distance(word, e)
            if d < bd: best, bd = e, d
            if bd == 1: return best, bd
    return best, bd

def lex_snap(word, tau):
    e, d = nearest(word)
    return e if (e is not None and d / max(len(word), 1) <= tau) else word

lexicon size: 1210


## 2. CRNN predictions + confidence

In [4]:
class Vocab:
    BLANK = 0
    def __init__(self, texts):
        chars = sorted(set("".join(texts)))
        self.idx2char = {i + 1: c for i, c in enumerate(chars)}
        self.char2idx = {c: i for i, c in self.idx2char.items()}
    def __len__(self): return len(self.idx2char) + 1
    def decode_conf(self, log_probs_row):
        probs = log_probs_row.exp(); idx = log_probs_row.argmax(-1).tolist()
        out, prev, confs = [], None, []
        for t, i in enumerate(idx):
            if i != prev and i != self.BLANK:
                out.append(self.idx2char[i]); confs.append(float(probs[t, i]))
            prev = i
        return "".join(out), (float(np.mean(confs)) if confs else 0.0)

class CRNN(nn.Module):
    def __init__(self, n, rnn_hidden=256, rnn_layers=2, dropout=0.2):
        super().__init__()
        def conv(i, o, bn=False):
            L = [nn.Conv2d(i, o, 3, 1, 1)]
            if bn: L.append(nn.BatchNorm2d(o))
            L.append(nn.ReLU(inplace=True)); return L
        self.cnn = nn.Sequential(*conv(1,64), nn.MaxPool2d(2,2), *conv(64,128), nn.MaxPool2d(2,2),
            *conv(128,256), *conv(256,256), nn.MaxPool2d((2,1),(2,1)),
            *conv(256,512,bn=True), *conv(512,512,bn=True), nn.MaxPool2d((2,1),(2,1)))
        self.collapse = nn.AdaptiveAvgPool2d((1, None))
        self.rnn = nn.LSTM(512, rnn_hidden, rnn_layers, bidirectional=True,
                           dropout=dropout if rnn_layers > 1 else 0.0)
        self.head = nn.Linear(2*rnn_hidden, n)
    def forward(self, x):
        f = self.collapse(self.cnn(x)).squeeze(2).permute(2, 0, 1)
        seq, _ = self.rnn(f); return self.head(seq)

class GreyDataset(Dataset):
    def __init__(self, csv, cfg):
        self.df = pd.read_csv(csv).dropna(subset=[cfg.label_col])
        self.df[cfg.label_col] = self.df[cfg.label_col].astype(str).str.strip()
        self.cfg = cfg; self.img_dir = cfg.data_root / cfg.img_dir
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(self.img_dir / str(row[self.cfg.img_col])).convert("L")
        w, h = img.size
        nw = min(max(1, int(round(w*self.cfg.img_height/h))), self.cfg.img_width)
        img = img.resize((nw, self.cfg.img_height), Image.BILINEAR)
        canvas = Image.new("L", (self.cfg.img_width, self.cfg.img_height), 255); canvas.paste(img, (0,0))
        x = torch.from_numpy(np.array(canvas, dtype=np.float32)/255.0).unsqueeze(0)
        return x, row[self.cfg.label_col].lower(), str(row[self.cfg.img_col])

def grey_collate(b):
    xs, t, f = zip(*b); return torch.stack(xs), list(t), list(f)

ck = torch.load(CFG.crnn_ckpt, map_location="cpu")
_tr = pd.read_csv(CFG.data_root / CFG.train_csv)[CFG.label_col].dropna().astype(str).str.strip().tolist()
VOCAB = Vocab(_tr)
assert len(VOCAB) == ck["model"]["head.bias"].shape[0], "CRNN vocab/checkpoint mismatch"
crnn = CRNN(len(VOCAB), CFG.rnn_hidden, CFG.rnn_layers, CFG.dropout); crnn.load_state_dict(ck["model"])
crnn.to(DEVICE).eval()

@torch.no_grad()
def crnn_predict(csv):
    dl = DataLoader(GreyDataset(csv, CFG), CFG.batch_size, shuffle=False,
                    num_workers=CFG.num_workers, collate_fn=grey_collate)
    res = {}
    for xb, texts, fnames in dl:
        lp = crnn(xb.to(DEVICE)).log_softmax(-1).permute(1, 0, 2).cpu()
        for row, t, f in zip(lp, texts, fnames):
            s, c = VOCAB.decode_conf(row)
            res[f] = {"raw": s, "conf": c, "ref": t,
                      "lex": lex_snap(s, CFG.crnn_lex_tau)}
    return res

print("running CRNN...")
crnn_val = crnn_predict(CFG.data_root / CFG.val_csv)
crnn_test = crnn_predict(CFG.data_root / CFG.test_csv)
print(f"CRNN done: {len(crnn_test)} test preds")

running CRNN...
CRNN done: 791 test preds


## 3. TrOCR predictions + confidence

In [5]:
processor = TrOCRProcessor.from_pretrained(CFG.processor_name)
trocr = VisionEncoderDecoderModel.from_pretrained(CFG.trocr_ckpt).to(DEVICE).eval()

class RGBDataset(Dataset):
    def __init__(self, csv, cfg, processor):
        self.df = pd.read_csv(csv).dropna(subset=[cfg.label_col]).reset_index(drop=True)
        self.df[cfg.label_col] = self.df[cfg.label_col].astype(str).str.strip()
        self.cfg = cfg; self.processor = processor; self.img_dir = cfg.data_root / cfg.img_dir
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(self.img_dir / str(row[self.cfg.img_col])).convert("RGB")
        pv = self.processor(img, return_tensors="pt").pixel_values.squeeze(0)
        return {"pixel_values": pv, "text": row[self.cfg.label_col].lower(), "fname": str(row[self.cfg.img_col])}

def rgb_collate(b):
    return {"pixel_values": torch.stack([x["pixel_values"] for x in b]),
            "text": [x["text"] for x in b], "fname": [x["fname"] for x in b]}

@torch.no_grad()
def trocr_predict(csv):
    dl = DataLoader(RGBDataset(csv, CFG, processor), CFG.batch_size, shuffle=False,
                    num_workers=CFG.num_workers, collate_fn=rgb_collate)
    res = {}
    for batch in dl:
        out = trocr.generate(batch["pixel_values"].to(DEVICE), max_new_tokens=CFG.max_target_len,
                             output_scores=True, return_dict_in_generate=True)
        ts = trocr.compute_transition_scores(out.sequences, out.scores, normalize_logits=True)
        for i in range(out.sequences.size(0)):
            text = processor.decode(out.sequences[i], skip_special_tokens=True).strip().lower()
            valid = ts[i][torch.isfinite(ts[i])]
            conf = float(valid.mean().exp()) if valid.numel() else 0.0
            res[batch["fname"][i]] = {"raw": text, "conf": conf, "ref": batch["text"][i]}
    return res

print("running TrOCR...")
trocr_val = trocr_predict(CFG.data_root / CFG.val_csv)
trocr_test = trocr_predict(CFG.data_root / CFG.test_csv)
print(f"TrOCR done: {len(trocr_test)} test preds")

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

running TrOCR...
TrOCR done: 791 test preds


## 4. Fusion referee
Per word: (1) agree -> use; (2) disagree & exactly one is an exact lexicon hit -> prefer it;
(3) else -> trust the more confident model. `trocr_bias` slightly favours TrOCR on ties since
it is the stronger base model (tuned on validation).

In [6]:
def fuse(crnn_res, trocr_res, trocr_bias=0.0):
    """TrOCR-anchored referee. Start from TrOCR + lexicon. Override with the CRNN's answer
    ONLY when the CRNN+lexicon produced an EXACT lexicon match that TrOCR's (lexicon-snapped)
    output did not — i.e. the CRNN confirms a real drug name TrOCR missed. This avoids the
    invalid cross-model confidence comparison (TrOCR and CRNN confidences are on different
    scales) that made naive fusion over-trust the weaker model.

    `trocr_bias` is retained for signature compatibility but unused (no confidence comparison).
    """
    preds, refs, files = [], [], sorted(set(crnn_res) & set(trocr_res))
    for f in files:
        c = crnn_res[f]; t = trocr_res[f]
        t_lex = lex_snap(t["raw"], CFG.crnn_lex_tau)     # TrOCR + lexicon = the base
        c_out = c["lex"]                                  # CRNN + lexicon
        # override only when CRNN gives a confirmed lexicon hit and TrOCR (snapped) does not
        if (c_out in LEXSET) and (t_lex not in LEXSET):
            chosen = c_out
        else:
            chosen = t_lex
        preds.append(chosen); refs.append(t["ref"])
    return preds, refs, files

# The TrOCR-anchored referee has no confidence comparison to tune; verify on validation that
# fusion does not fall below TrOCR+lexicon there.
val_files = sorted(set(crnn_val) & set(trocr_val))
val_trocr_lex = [lex_snap(trocr_val[f]["raw"], CFG.crnn_lex_tau) for f in val_files]
val_refs_l = [trocr_val[f]["ref"] for f in val_files]
val_fuse, val_fref, _ = fuse(crnn_val, trocr_val)
best_bias = 0.0
print(f"VALIDATION — TrOCR+lex EM {corpus_metrics(val_trocr_lex, val_refs_l)['ExactMatch']:.4f} "
      f"| fusion EM {corpus_metrics(val_fuse, val_fref)['ExactMatch']:.4f}")

VALIDATION — TrOCR+lex EM 0.6241 | fusion EM 0.6253


## 5. Test: all models + fusion

In [7]:
# rebuild each model's test predictions as lists aligned by file
files = sorted(set(crnn_test) & set(trocr_test))
refs = [trocr_test[f]["ref"] for f in files]
crnn_lex_list = [crnn_test[f]["lex"] for f in files]
trocr_list = [trocr_test[f]["raw"] for f in files]
trocr_lex_list = [lex_snap(trocr_test[f]["raw"], CFG.crnn_lex_tau) for f in files]
fuse_list, fuse_refs, _ = fuse(crnn_test, trocr_test, best_bias)

print(f"TEST (n={len(files)}):")
for name, preds in [("CRNN + lexicon", crnn_lex_list),
                    ("TrOCR", trocr_list),
                    ("TrOCR + lexicon", trocr_lex_list),
                    ("FUSION (CRNN+TrOCR+lexicon)", fuse_list)]:
    m = corpus_metrics(preds, refs)
    print(f"  {name:32s}: CER {m['CER']:.4f} | EM {m['ExactMatch']:.4f}")

# seen/unseen for fusion
test_meta = pd.read_csv(CFG.data_root / CFG.test_csv)
seen_map = dict(zip(test_meta[CFG.img_col].astype(str), test_meta["seen_in_train"]))
groups = {"seen": ([], []), "unseen": ([], [])}
for p, r, f in zip(fuse_list, refs, files):
    k = "seen" if seen_map.get(f, False) else "unseen"
    groups[k][0].append(p); groups[k][1].append(r)
print("FUSION seen/unseen:")
for kk, (P, R) in groups.items():
    if R:
        m = corpus_metrics(P, R)
        print(f"  {kk:6s} (n={m['n']:3d}): EM {m['ExactMatch']:.4f}")

# how often each source was chosen / agreement rate
agree = sum(1 for f in files if crnn_test[f]["lex"] == trocr_test[f]["raw"])
print(f"\nmodels agreed on {agree}/{len(files)} ({agree/len(files):.1%}) words")

pd.DataFrame([{"model": "CRNN+lex", **corpus_metrics(crnn_lex_list, refs)},
              {"model": "TrOCR", **corpus_metrics(trocr_list, refs)},
              {"model": "TrOCR+lex", **corpus_metrics(trocr_lex_list, refs)},
              {"model": "Fusion", **corpus_metrics(fuse_list, refs)}]
             ).to_csv(CFG.out_dir / "fusion_results.csv", index=False)

TEST (n=791):
  CRNN + lexicon                  : CER 0.4903 | EM 0.3047
  TrOCR                           : CER 0.2159 | EM 0.5689
  TrOCR + lexicon                 : CER 0.2343 | EM 0.6119
  FUSION (CRNN+TrOCR+lexicon)     : CER 0.2340 | EM 0.6094
FUSION seen/unseen:
  seen   (n=615): EM 0.7837
  unseen (n=176): EM 0.0000

models agreed on 227/791 (28.7%) words


## 6. For the thesis
- If FUSION > TrOCR+lexicon: genuine novelty — "a confidence- and lexicon-aware fusion of a
  pretrained transformer and a lexicon-augmented CRNN exceeds either model alone." Report the
  per-source selection rate and the referee rules as the contribution.
- If FUSION ties TrOCR+lexicon: honest finding — "the stronger base model dominates; fusion
  does not exceed it because the CRNN rarely contributes a correction TrOCR lacks." Still a
  valid, reportable result, and the agreement-rate analysis explains WHY.
- Either way report the agreement rate and seen/unseen breakdown — they characterise exactly
  when each model contributes.